In [1]:
# pip install lightgbm scikit-learn pandas

In [1]:
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display
from itertools import product

from download_dataset import YambdaDataset
from split_dataset import flat_split_train_val_test_pd
from markov_chain import MarkovChain
from constants import Constants
from create_data_for_train import create_dataset_for_train

/Users/leonidlevin/jupyter_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Загружаем датасет с выбранными параметрами

In [2]:
dataset = YambdaDataset("flat", "50m")
events = dataset.interaction("multi_event")
df = events.to_pandas()

unique_users = df['uid'].unique()
sampled_users = pd.Series(unique_users).sample(n=100, random_state=42)  # random_state для воспроизводимости

# Фильтруем исходный датафрейм, оставляя только выбранных пользователей
df = df[df['uid'].isin(sampled_users)]

## Разделяем на выборки для обучения и тестирования

In [3]:
train, val, test = flat_split_train_val_test_pd(df, test_timestamp=Constants.TEST_TIMESTAMP, val_size=0, gap_size=Constants.GAP_SIZE)

In [4]:
mh = MarkovChain(
    name='Yambda_Markov',
    data=train,
    struct=('event_type', 'item_id'),
    time_column='timestamp',
    users_id_column='uid'
)
mh.preprocessing_data()
mh.build_markov_chain()

In [5]:
filtered_train_like = create_dataset_for_train(df, mh, Constants.DEPTH, last='like', pre_last='listen')
filtered_train_dislike = create_dataset_for_train(df, mh, Constants.DEPTH, last='dislike', pre_last='listen')

In [6]:
filtered_train_like['target'] = 1
filtered_train_dislike['target'] = 0

In [9]:
filtered_train_like = filtered_train_like[:400]

In [10]:
train_data = pd.concat([filtered_train_like, filtered_train_dislike])

In [11]:
train_df = train_data.copy()
probs = [f'P{i}{i+1}' for i in range(1, Constants.DEPTH)]

for i in range(1, len(probs)+1):
    train_df[f'P{i}{i+1}'] = train_df[probs[:i]].prod(axis=1)

# Оставляем только новые колонки и таргет
needed_columns = [f'P{i}{i+1}' for i in range(1, len(probs)+1)] + ['target']
train_df = train_df[needed_columns]

In [12]:
train_df.head()

,P12,P23,P34,P45,P56,P67,P78,P89,P910,target
0,0.035714,0.000616,9.163148e-07,1.799213e-13,3.625628e-24,4.107869e-49,4.778644e-98,6.974065e-195,0.000000e+00,1
1,0.008065,0.000050,2.693019e-07,9.712974e-16,1.089308e-30,1.798429e-60,3.234345e-120,6.695034e-238,0.000000e+00,1
3,0.083333,0.062500,1.186408e-05,4.413722e-09,8.264642e-18,6.728484e-36,1.516629e-68,2.300163e-136,2.645375e-272,1
5,1.000000,0.038462,9.615385e-03,4.350853e-06,5.548425e-11,8.927655e-20,6.641919e-40,5.293810e-78,3.503053e-156,1
6,0.500000,0.023810,1.190476e-02,4.168334e-06,8.439290e-11,1.246378e-20,4.142556e-41,1.287058e-80,1.325215e-161,1


In [16]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import lightgbm as lgb

# Загружаем данные
df = train_df  # путь к твоему файлу

# Указываем признаки
features = ['P12', 'P23', 'P34', 'P45', 'P56', 'P67', 'P78', 'P89', 'P910']
X = df[features]
y = df['target']

# Делим на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [17]:
print(X_train.nunique())  # количество уникальных значений по каждой фиче


P12     169
P23     449
P34     576
P45     613
P56     626
P67     630
P78     632
P89     635
P910    439
dtype: int64


In [17]:
# Обучаем модель
model = lgb.LGBMClassifier(n_estimators=10000, learning_rate=0.01)
model.fit(X_train, y_train)

# Предсказания
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Оценка
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

  Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl.metadata (17 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 7.4 MB/s eta 0:00:007.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 7.2 MB/s eta 0:00:007.4 MB/s eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [lightgbm]━━ 3/5 [scikit-learn]
Note: you may need to restart the kernel to use updated packages.
